# Retail Supply Chain Analytics

## Predictive Analytics & Sales Forecasting

This notebook develops machine learning models to forecast weekly retail sales using the processed dataset created during the data preparation phase.

The workflow includes:

- Loading the processed dataset
- Feature selection
- Time-aware train/test splitting
- Model training
- Model evaluation
- Feature importance analysis
- Business interpretation

## Importing Libraries

The required Python libraries are imported for data manipulation, model development, evaluation and visualisation.

In [14]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Loading the Processed Dataset

The modelling dataset generated during the feature engineering phase is loaded.

This dataset contains cleaned observations together with engineered calendar, sales and business features.

In [15]:
# =====================================================
# LOAD PROCESSED DATASET
# =====================================================

model_df = pd.read_csv(
    "../data/processed/model_dataset.csv"
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## Dataset Overview

Before modelling begins, the processed dataset is reviewed to confirm its dimensions, variables and data types.

In [16]:
# =====================================================
# DATASET OVERVIEW
# =====================================================

display(model_df.head())

print("Shape:", model_df.shape)

,store_id,department,date,weekly_sales,is_holiday,temperature,fuel_price,markdown_1,markdown_2,markdown_3,...,cumulative_sales,rolling_4_week_std,season_Spring,season_Summer,season_Winter,store_type_B,store_type_C,region_North,region_South,region_West
0,1,1,2022-01-08,35525.05,0,46.03,3.67,1356.75,2486.21,1427.01,...,154601.01,59079.415035,0,0,1,0,0,1,0,0
1,1,1,2022-01-15,14847.56,0,25.96,5.46,3861.22,596.15,22.09,...,169448.57,55184.346393,0,0,1,0,0,1,0,0
2,1,1,2022-01-22,45254.57,0,25.92,3.58,579.35,2589.31,2493.19,...,214703.14,45406.240356,0,0,1,0,0,1,0,0
3,1,1,2022-01-29,15166.69,0,78.37,4.41,4436.06,1416.64,478.38,...,229869.83,15184.020103,0,0,1,0,0,1,0,0
4,1,1,2022-02-05,15601.13,0,59.50,4.07,2137.71,76.26,431.57,...,245470.96,15027.895544,0,0,1,0,0,1,0,0


Shape: (155000, 32)


## Preparing the Date Column

The processed dataset was loaded from a CSV file, causing the `date` column to be stored as a string.

The column is converted back to the datetime format so that observations can be sorted chronologically and used for time-based model training and evaluation.

In [17]:
# =====================================================
# CONVERT DATE COLUMN TO DATETIME
# =====================================================

model_df["date"] = pd.to_datetime(model_df["date"])

print("Date converted successfully.")

print("\nDate Range:")
print("Start:", model_df["date"].min())
print("End  :", model_df["date"].max())

Date converted successfully.

Date Range:
Start: 2022-01-08 00:00:00
End  : 2024-12-21 00:00:00


## Sorting the Dataset

Time-series forecasting requires observations to remain in chronological order.

The dataset is sorted by date, store and department before splitting into training and testing data.

In [18]:
# =====================================================
# SORT DATASET
# =====================================================

model_df = model_df.sort_values(
    by=["date", "store_id", "department"]
).reset_index(drop=True)

display(model_df.head())

,store_id,department,date,weekly_sales,is_holiday,temperature,fuel_price,markdown_1,markdown_2,markdown_3,...,cumulative_sales,rolling_4_week_std,season_Spring,season_Summer,season_Winter,store_type_B,store_type_C,region_North,region_South,region_West
0,1,1,2022-01-08,35525.05,0,46.03,3.67,1356.75,2486.21,1427.01,...,154601.01,59079.415035,0,0,1,0,0,1,0,0
1,1,2,2022-01-08,64660.95,0,46.03,3.67,1356.75,2486.21,1427.01,...,183768.80,38499.772205,0,0,1,0,0,1,0,0
2,1,3,2022-01-08,72545.36,0,46.03,3.67,1356.75,2486.21,1427.01,...,156915.24,8361.198276,0,0,1,0,0,1,0,0
3,1,4,2022-01-08,80907.98,0,46.03,3.67,1356.75,2486.21,1427.01,...,169353.22,5329.647658,0,0,1,0,0,1,0,0
4,1,5,2022-01-08,79834.36,0,46.03,3.67,1356.75,2486.21,1427.01,...,144994.21,10376.445532,0,0,1,0,0,1,0,0


In [19]:
# =====================================================
# DATA TYPES
# =====================================================

model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 155000 entries, 0 to 154999
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   store_id                 155000 non-null  int64         
 1   department               155000 non-null  int64         
 2   date                     155000 non-null  datetime64[us]
 3   weekly_sales             155000 non-null  float64       
 4   is_holiday               155000 non-null  int64         
 5   temperature              155000 non-null  float64       
 6   fuel_price               155000 non-null  float64       
 7   markdown_1               155000 non-null  float64       
 8   markdown_2               155000 non-null  float64       
 9   markdown_3               155000 non-null  float64       
 10  markdown_4               155000 non-null  float64       
 11  markdown_5               155000 non-null  float64       
 12  cpi                      15

In [20]:
# =====================================================
# CHECK MISSING VALUES
# =====================================================

model_df.isnull().sum().sort_values(ascending=False)

store_id                   0
department                 0
date                       0
weekly_sales               0
is_holiday                 0
temperature                0
fuel_price                 0
markdown_1                 0
markdown_2                 0
markdown_3                 0
markdown_4                 0
markdown_5                 0
cpi                        0
unemployment               0
store_size                 0
year                       0
month                      0
quarter                    0
week_number                0
previous_week_sales        0
rolling_4_week_avg         0
weekly_sales_growth_pct    0
cumulative_sales           0
rolling_4_week_std         0
season_Spring              0
season_Summer              0
season_Winter              0
store_type_B               0
store_type_C               0
region_North               0
region_South               0
region_West                0
dtype: int64

## Defining the Target Variable

The objective of the forecasting models is to predict weekly sales.

The `weekly_sales` column is therefore selected as the target variable, while all remaining predictive variables are used as model features.

In [21]:
# =====================================================
# DEFINE TARGET VARIABLE
# =====================================================

target = "weekly_sales"

print(f"Target Variable: {target}")

Target Variable: weekly_sales


## Selecting Predictor Variables

The target variable (`weekly_sales`) is separated from the predictor variables.

Features that would introduce target leakage or are used only for chronological ordering are excluded from the modelling dataset.

In [22]:
# =====================================================
# DEFINE FEATURES AND TARGET
# =====================================================

target = "weekly_sales"

excluded_features = [
    "weekly_sales",
    "date",
    "cumulative_sales",
    "weekly_sales_growth_pct"
]

feature_columns = [
    column
    for column in model_df.columns
    if column not in excluded_features
]

X = model_df[feature_columns]
y = model_df[target]

print(f"Number of Features: {X.shape[1]}")
print(f"Number of Observations: {X.shape[0]}")

print("\nSelected Features:")
print(feature_columns)

Number of Features: 28
Number of Observations: 155000

Selected Features:
['store_id', 'department', 'is_holiday', 'temperature', 'fuel_price', 'markdown_1', 'markdown_2', 'markdown_3', 'markdown_4', 'markdown_5', 'cpi', 'unemployment', 'store_size', 'year', 'month', 'quarter', 'week_number', 'previous_week_sales', 'rolling_4_week_avg', 'rolling_4_week_std', 'season_Spring', 'season_Summer', 'season_Winter', 'store_type_B', 'store_type_C', 'region_North', 'region_South', 'region_West']


## Time-Based Train/Test Split

Unlike traditional machine learning problems, forecasting models must respect the chronological order of observations.

Instead of randomly splitting the dataset, the earliest observations are used for training while the most recent observations are reserved for testing.

This approach prevents data leakage and provides a realistic estimate of forecasting performance.

In [23]:
# =====================================================
# CHECK AVAILABLE DATE RANGE
# =====================================================

print("Start Date :", model_df["date"].min())
print("End Date   :", model_df["date"].max())

Start Date : 2022-01-08 00:00:00
End Date   : 2024-12-21 00:00:00


In [24]:
# =====================================================
# DETERMINE SPLIT DATE
# =====================================================

split_index = int(len(model_df) * 0.80)

split_date = model_df.iloc[split_index]["date"]

print("Training ends at :", split_date)

Training ends at : 2024-05-25 00:00:00


### Creating Training and Testing Datasets

The processed data is divided chronologically into training and testing sets.

The training set contains approximately 80% of the earliest observations and is used for model development.

The remaining 20% represents unseen future observations and is reserved for evaluating forecasting performance.

In [25]:
# =====================================================
# CREATE TRAIN AND TEST SETS
# =====================================================

train = model_df[model_df["date"] < split_date]

test = model_df[model_df["date"] >= split_date]

print("Training Shape :", train.shape)
print("Testing Shape  :", test.shape)

print("\nTraining Period")
print(train["date"].min(), "to", train["date"].max())

print("\nTesting Period")
print(test["date"].min(), "to", test["date"].max())

Training Shape : (124000, 32)
Testing Shape  : (31000, 32)

Training Period
2022-01-08 00:00:00 to 2024-05-18 00:00:00

Testing Period
2024-05-25 00:00:00 to 2024-12-21 00:00:00


### Preparing Model Inputs

The predictor variables and target variable are separated for both the training and testing datasets.

This creates the input matrices (`X`) and target vectors (`y`) required for model training.

In [26]:
# =====================================================
# SPLIT FEATURES AND TARGET
# =====================================================

X_train = train[feature_columns]
X_test = test[feature_columns]

y_train = train[target]
y_test = test[target]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (124000, 28)
X_test : (31000, 28)
y_train: (124000,)
y_test : (31000,)
